## Section 1: Overview
This section outlines goals and the LoRA approach.

# EoMT + LoRA Fine-tuning (Cityscapes)

Goal: Improve anomaly segmentation while preserving cross-dataset robustness using Low-Rank Adaptation (LoRA).

Approach:
- Freeze ViT encoder + EoMT decoder base weights
- Inject LoRA adapters into attention projections (Q/K/V) and optionally MLP
- Train only LoRA params + prediction heads
- Keep Logit Normalization loss for calibration

Why: Head-only fine-tuning improved in-domain but hurt robustness. LoRA limits trainable DOF to reduce drift while allowing controlled adaptation.

## Section 2: Environment and Imports
Setup the working directory, project paths, and import EoMT components.

In [ ]:
# Environment & Imports
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from datetime import datetime
from tqdm import tqdm
import csv

# Detect Colab for optional installs
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    # Minimal deps if needed
    !pip -q install lightning timm

# Project base
REPO_BASE = Path('..')
os.chdir(REPO_BASE / 'eomt')
print('Working directory:', os.getcwd())

# Bring project modules into path
sys.path.insert(0, str(REPO_BASE / 'eomt'))
from models.eomt import EoMT
from models.vit import ViT
from training.mask_classification_semantic import MaskClassificationSemantic
from datasets.cityscapes_semantic import CityscapesSemantic
print('✓ Imports ready')

## Section 3: LoRA Utilities
Helper modules to wrap attention projections with low-rank adapters.

In [ ]:
# LoRA utilities
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16, dropout_p: float = 0.0):
        super().__init__()
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        # Freeze base
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        # LoRA factors (low-rank)
        if r > 0:
            self.lora_A = nn.Linear(self.in_features, r, bias=False)
            self.lora_B = nn.Linear(r, self.out_features, bias=False)
            # Init per LoRA paper
            nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B.weight)
            self.scaling = self.alpha / self.r
        else:
            self.lora_A = None
            self.lora_B = None
            self.scaling = 0.0

    def forward(self, x):
        result = self.base(x)
        if self.r > 0:
            return result + self.dropout(self.lora_B(self.lora_A(x))) * self.scaling
        return result

import math

def replace_linear_with_lora(module: nn.Module, r=8, alpha=16, dropout_p=0.0, target_names=("qkv", "proj")):
    """
    Recursively wrap nn.Linear layers in attention blocks (e.g., timm ViT) with LoRA.
    By default, targets typical attention names: 'qkv', 'proj'.
    """
    for name, child in list(module.named_children()):
        # Heuristic: only wrap lines inside attention or projection paths
        lname = name.lower()
        if isinstance(child, nn.Linear) and any(t in lname for t in target_names):
            setattr(module, name, LoRALinear(child, r=r, alpha=alpha, dropout_p=dropout_p))
        else:
            replace_linear_with_lora(child, r=r, alpha=alpha, dropout_p=dropout_p, target_names=target_names)

def lora_parameters_only(model: nn.Module):
    for n, p in model.named_parameters():
        if any(s in n for s in ['lora_A.weight', 'lora_B.weight']):
            yield p

## Section 4: Configuration
LoRA and training hyperparameters, paths, and output directory.

In [ ]:
# Configuration
CONFIG = {
    'epochs': 2,
    'batch_size': 1,
    'grad_accum': 4,
    'lr_lora': 1e-4,
    'lr_heads': 5e-5,
    'weight_decay': 1e-4,
    'use_amp': True,
    'num_workers': 2,
    'img_size': 1024,
    'freeze_encoder': True,
    'freeze_decoder': True,
    'lora_r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.0,
    'cityscapes_root': str(REPO_BASE / 'dataset' / 'cityscapes'),
    'pretrained_ckpt': str(REPO_BASE / 'checkpoints' / 'eomt_cityscapes.bin'),
    'output_dir': str(REPO_BASE / 'checkpoints' / 'lora_fine_tuned'),
}
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
print('✓ Config ready')

## Section 5: Logit Normalization Loss
Loss for confidence calibration used during LoRA fine-tuning.

In [ ]:
# Loss: Logit Normalization
class LogitNormalizationLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    def forward(self, logits, targets, mask=None, ignore_index=None):
        logits_norm = F.normalize(logits, p=2, dim=1) / self.temperature
        if ignore_index is not None:
            loss = F.cross_entropy(logits_norm, targets, reduction='none', ignore_index=ignore_index)
        else:
            loss = F.cross_entropy(logits_norm, targets, reduction='none')
        if mask is not None:
            loss = (loss * mask).sum() / mask.sum().clamp(min=1)
        else:
            loss = loss.mean()
        return loss

criterion = LogitNormalizationLoss(temperature=0.07)
print('✓ Loss ready')

## Section 6: Model and LoRA Injection
Load pretrained EoMT, freeze base modules, and inject LoRA adapters.

In [ ]:
# Build model and inject LoRA
NUM_CLASSES = 19
MODEL_IMG_SIZE = (CONFIG['img_size'], CONFIG['img_size'])
PATCH_SIZE = 16
NUM_QUERIES = 100
NUM_BLOCKS = 3
BACKBONE_NAME = 'vit_base_patch14_reg4_dinov2'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

encoder = ViT(img_size=MODEL_IMG_SIZE, patch_size=PATCH_SIZE, backbone_name=BACKBONE_NAME)
network = EoMT(encoder=encoder, num_classes=NUM_CLASSES, num_q=NUM_QUERIES, num_blocks=NUM_BLOCKS, masked_attn_enabled=True)
model = MaskClassificationSemantic(
    network=network,
    img_size=MODEL_IMG_SIZE,
    num_classes=NUM_CLASSES,
    attn_mask_annealing_enabled=False,
    attn_mask_annealing_start_steps=None,
    attn_mask_annealing_end_steps=None,
    ckpt_path=CONFIG['pretrained_ckpt'],
    delta_weights=False,
    load_ckpt_class_head=True,
)
model = model.to(device)
print('✓ Pretrained model loaded')

# Freeze encoder/decoder base weights
def freeze_module(m, freeze=True):
    for p in m.parameters():
        p.requires_grad = not freeze
if CONFIG['freeze_encoder']:
    freeze_module(model.network.encoder, True)
if CONFIG['freeze_decoder'] and hasattr(model.network, 'decoder'):
    freeze_module(model.network.decoder, True)

# Inject LoRA into encoder attention projections
replace_linear_with_lora(model.network.encoder, r=CONFIG['lora_r'], alpha=CONFIG['lora_alpha'], dropout_p=CONFIG['lora_dropout'])

# Ensure heads remain trainable
for n, p in model.named_parameters():
    if 'class_head' in n or 'mask_head' in n:
        p.requires_grad = True

# Build optimizers: separate LR for LoRA vs heads
lora_params = list(lora_parameters_only(model))
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'lora_' not in n]
optimizer = torch.optim.AdamW([
    {'params': lora_params, 'lr': CONFIG['lr_lora']},
    {'params': head_params, 'lr': CONFIG['lr_heads']}
], weight_decay=CONFIG['weight_decay'])
scaler = GradScaler() if CONFIG['use_amp'] else None

print(f'✓ Trainable params (LoRA + heads): {sum(p.numel() for p in lora_params + head_params):,}')

## Section 7: Cityscapes Dataloader
Build dataloaders directly from zipped Cityscapes archives.

In [ ]:
# Data: Cityscapes from zips
zip_dir = Path(CONFIG['cityscapes_root'])
dm = CityscapesSemantic(
    path=zip_dir,
    num_workers=CONFIG['num_workers'],
    batch_size=CONFIG['batch_size'],
    img_size=(CONFIG['img_size'], CONFIG['img_size']),
    num_classes=NUM_CLASSES,
    color_jitter_enabled=False,
    check_empty_targets=True,
)
dm.setup(stage='fit')
train_loader = dm.train_dataloader()
print('✓ DataLoader ready:', len(train_loader), 'batches')

## Section 8: Training Utilities
Logging, AMP training loop with gradient accumulation, and helpers.

In [ ]:
# Training utils
csv_path = Path(CONFIG['output_dir']) / 'training_loss_lora.csv'
def log_loss(epoch, batch, loss_val):
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch','batch','loss','timestamp'])
        if f.tell() == 0:
            writer.writeheader()
        writer.writerow({'epoch':epoch,'batch':batch,'loss':float(loss_val),'timestamp':datetime.now().isoformat()})

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss, num_batches = 0.0, len(dataloader)
    ignore_idx = getattr(model, 'ignore_idx', 255)
    accum = CONFIG['grad_accum']
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}')
    for batch_idx, batch in enumerate(pbar):
        images, targets = batch
        images = images.to(device)
        per_pixel_targets = model.to_per_pixel_targets_semantic(targets, ignore_idx)
        per_pixel_targets = torch.stack(per_pixel_targets).to(device).long()
        target_hw = per_pixel_targets.shape[-2:]
        invalid_mask = (per_pixel_targets < 0) | ((per_pixel_targets > 18) & (per_pixel_targets != ignore_idx))
        if invalid_mask.any():
            per_pixel_targets = torch.where(invalid_mask, ignore_idx, per_pixel_targets)
        valid_mask = (per_pixel_targets != ignore_idx).float().to(device)
        if scaler is not None:
            try:
                with torch.amp.autocast('cuda'):
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx) / accum
            except AttributeError:
                with autocast(enabled=True):
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx) / accum
            scaler.scale(loss).backward()
        else:
            mask_logits_list, class_logits_list = model(images)
            mask_logits = mask_logits_list[-1]
            class_logits = class_logits_list[-1]
            per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
            loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx) / accum
            loss.backward()
        if (batch_idx + 1) % accum == 0 or (batch_idx + 1) == num_batches:
            if scaler is not None:
                scaler.step(optimizer); scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
        total_loss += loss.item() * accum
        avg_loss = total_loss / (batch_idx + 1)
        log_loss(epoch, batch_idx, loss.item() * accum)
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    return total_loss / num_batches

print('✓ Training utilities ready')

## Section 9: Run Training
Train LoRA + heads with mixed precision and save checkpoints.

In [ ]:
# Run training
if train_loader is not None:
    best_loss = float('inf')
    start = datetime.now()
    for epoch in range(CONFIG['epochs']):
        loss_val = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch)
        print(f'Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {loss_val:.4f}')
        ckpt_path = Path(CONFIG['output_dir']) / f'eomt_lora_epoch{epoch+1}.pth'
        torch.save(model.state_dict(), ckpt_path)
        if loss_val < best_loss:
            best_loss = loss_val
            torch.save(model.state_dict(), Path(CONFIG['output_dir']) / 'eomt_lora_best.pth')
    total_h = (datetime.now() - start).total_seconds() / 3600
    print(f'Training completed in {total_h:.2f}h; best loss {best_loss:.4f}')
else:
    print('DataLoader unavailable')

## Section 10: Next Steps
How to evaluate LoRA checkpoints and iterate on hyperparameters.

## Next Steps
- Evaluate with evalAnomaly.py using the LoRA checkpoint (eomt_lora_best.pth) on all datasets.
- Compare against head-only fine-tuning and baseline.
- If drift persists, reduce r (e.g., 4), lower LR, or restrict LoRA to Q/K only.

## Section 11: Evaluation (LoRA vs Baseline)
Evaluate the LoRA-fine-tuned checkpoint against the baseline EoMT on all available anomaly datasets using MaxEntropy. Results are summarized and saved to CSV in the output directory.

In [ ]:
# Discover available anomaly datasets (Validation_Dataset)
from pathlib import Path

validation_base = REPO_BASE / "dataset" / "anomaly" / "Validation_Dataset"
print("Checking anomaly datasets in Validation_Dataset...")
print(f"Base path: {validation_base}")

available_datasets = {}
if validation_base.exists():
    for dataset_dir in sorted(validation_base.iterdir()):
        if dataset_dir.is_dir():
            images_path = dataset_dir / 'images'
            if images_path.exists():
                num_images = len(list(images_path.glob('*.*')))
                available_datasets[dataset_dir.name] = str(images_path / '*.*')
                print(f"✓ {dataset_dir.name:30s} ({num_images} images)")
else:
    print(f"✗ Validation_Dataset folder not found at: {validation_base}")

print(f"\nTotal available: {len(available_datasets)} datasets")
print("Datasets:")
for name in sorted(available_datasets.keys()):
    print(f"  - {name}")

In [ ]:
# Evaluation configuration
from pathlib import Path
EVAL = {
    'lora_ckpt': str(REPO_BASE / 'checkpoints' / 'lora_fine_tuned' / 'eomt_lora_best.pth'),
    'baseline_ckpt': str(REPO_BASE / 'checkpoints' / 'eomt_cityscapes.bin'),
    'method': 'maxentropy',
    'temperatures': [0.5, 0.75, 1.0, 1.1, 1.2, 2.0],
}
print('Evaluation config:')
for k, v in EVAL.items():
    print(' ', k, ':', v)

In [ ]:
# Run per-dataset evaluation for LoRA vs Baseline (MaxEntropy)
import subprocess, re
from collections import defaultdict


def run_eval(ckpt, dataset_path, method, temp):
    cmd = [
        'python', 'evalAnomaly.py',
        '--ckpt', ckpt,
        '--input', dataset_path,
        '--method', method,
        '--temp', str(temp)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        output = result.stdout + result.stderr
        auprc_match = re.search(r'AUPRC:\s*([\d.]+)', output)
        fpr_match = re.search(r'FPR@95TPR:\s*([\d.]+)', output)
        if auprc_match and fpr_match:
            return {
                'auprc': float(auprc_match.group(1)),
                'fpr95': float(fpr_match.group(1)),
                'success': True
            }
        return {'success': False, 'error': 'Parse error'}
    except subprocess.TimeoutExpired:
        return {'success': False, 'error': 'Timeout'}
    except Exception as e:
        return {'success': False, 'error': str(e)}

all_results = {}
for dataset_name in sorted(available_datasets.keys()):
    print('\n' + '='*100)
    print('DATASET:', dataset_name)
    print('='*100)
    dpath = available_datasets[dataset_name]
    lora_res, base_res = {}, {}
    for t in EVAL['temperatures']:
        print(f"[T={t}] LoRA...", end=' ')
        r1 = run_eval(EVAL['lora_ckpt'], dpath, EVAL['method'], t)
        if r1.get('success'):
            print(f"AUPRC: {r1['auprc']:.2f}%, FPR@95: {r1['fpr95']:.2f}%")
            lora_res[t] = r1
        else:
            print('FAILED:', r1.get('error'))
            lora_res[t] = None
        print(f"      Baseline...", end=' ')
        r2 = run_eval(EVAL['baseline_ckpt'], dpath, EVAL['method'], t)
        if r2.get('success'):
            print(f"AUPRC: {r2['auprc']:.2f}%, FPR@95: {r2['fpr95']:.2f}%")
            base_res[t] = r2
        else:
            print('FAILED:', r2.get('error'))
            base_res[t] = None
    all_results[dataset_name] = {'lora': lora_res, 'baseline': base_res}
print('\nEVALUATION DONE')

In [ ]:
# Summarize results and save CSV
import pandas as pd
from pathlib import Path

rows = []
for dataset_name, res in all_results.items():
    for t in EVAL['temperatures']:
        ft = res['lora'].get(t)
        bl = res['baseline'].get(t)
        if ft and bl:
            rows.append({
                'Dataset': dataset_name,
                'Temperature': t,
                'LoRA AUPRC': ft['auprc'],
                'Baseline AUPRC': bl['auprc'],
                'Improvement': ft['auprc'] - bl['auprc'],
                'LoRA FPR@95': ft['fpr95'],
                'Baseline FPR@95': bl['fpr95'],
            })

if rows:
    df = pd.DataFrame(rows)
    out_csv = Path(CONFIG['output_dir']) / f"eval_all_datasets_lora_{EVAL['method']}.csv"
    df.to_csv(out_csv, index=False)
    print('✓ Results saved:', out_csv)
    # Ranking by best improvement per dataset
    best_by_dataset = (
        df.sort_values('Improvement', ascending=False)
          .groupby('Dataset', as_index=False)
          .first()
          .sort_values('Improvement', ascending=False)
    )
    print('\nRANKING BY BEST IMPROVEMENT (LoRA vs Baseline)')
    for i, row in best_by_dataset.iterrows():
        print(f"- {row['Dataset']}: {row['Improvement']:+.2f}% @ T={row['Temperature']}")
else:
    print('No successful results to summarize.')